<a href="https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/01_rnaseq_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Workshop 1 — Exploring RNA-seq count data

**Companion to [Chapter 1: Brief Guide to RNA Sequencing](https://omics.iotok.org/chapters/rna-seq-basics)** on Omics Atlas.

In this hands-on you will load a **real** bulk RNA-seq dataset and take your first look at a gene-count matrix.

### The dataset — *airway* (Himes et al., 2014, PLoS ONE)
Four human airway smooth-muscle cell lines, each with an **untreated (control)** and a **dexamethasone-treated** sample (a glucocorticoid used in asthma). 8 samples, ~38,000 genes.

> Run each cell with **Shift+Enter**. Look for the ✏️ **Your turn** exercises.

## 1. Load the data
The count matrix (genes × samples) and the sample sheet live in the workshop repo — we read them straight from the web with pandas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

RAW = 'https://raw.githubusercontent.com/muntakson/omics-atlas-workshop/main'
counts = pd.read_csv(f'{RAW}/data/airway_scaledcounts.csv', index_col=0)
meta   = pd.read_csv(f'{RAW}/data/airway_metadata.csv', index_col=0)
print('counts (genes x samples):', counts.shape)
counts.head()

In [ ]:
# the sample sheet: which samples are control vs dexamethasone-treated
meta

## 2. Library sizes
The **library size** is the total number of mapped reads per sample. It varies between samples, which is exactly why raw counts must be *normalized* before comparison (Chapter 1 → *Alignment & Quantification*).

In [ ]:
libsize = counts.sum(axis=0) / 1e6  # millions of reads
ax = libsize.plot.bar(color=meta['dex'].map({'control':'#2f6fed','treated':'#12a594'}))
ax.set_ylabel('Library size (million reads)'); ax.set_title('Reads per sample')
plt.tight_layout(); plt.show()

## 3. Normalize: counts → CPM → log
**CPM** (counts per million) removes the effect of library size. We then `log2`-transform so the huge dynamic range of expression becomes readable.

In [ ]:
import numpy as np
cpm = counts / counts.sum(axis=0) * 1e6
logcpm = np.log2(cpm + 1)
logcpm.iloc[:, :3].plot.box(figsize=(5,4))
plt.ylabel('log2(CPM + 1)'); plt.title('Expression distribution (first 3 samples)')
plt.tight_layout(); plt.show()

### ✏️ Your turn
Find the **10 most highly expressed genes** across all samples (by total raw count). *Hint: `counts.sum(axis=1)` then `.sort_values`.*

In [ ]:
# your code here


<details><summary>▶ Solution</summary>

In [ ]:
counts.sum(axis=1).sort_values(ascending=False).head(10)

</details>

## 4. Do the samples separate? — PCA
Principal Component Analysis compresses ~38,000 genes into 2 dimensions. If the biology is strong, treated and control samples should split apart (Chapter 1 → *Quality Check of Samples*).

In [ ]:
from sklearn.decomposition import PCA
# keep the 2000 most variable genes, samples as rows
top = logcpm.var(axis=1).sort_values(ascending=False).head(2000).index
X = logcpm.loc[top].T
pcs = PCA(n_components=2).fit_transform(X)
pc = pd.DataFrame(pcs, columns=['PC1','PC2'], index=X.index).join(meta)
sns.scatterplot(data=pc, x='PC1', y='PC2', hue='dex', s=120)
plt.title('PCA of airway samples'); plt.tight_layout(); plt.show()

## ✅ Recap
You loaded real RNA-seq counts, inspected library sizes, normalized to log-CPM, and saw treated vs control separate by PCA.

**Next → [Workshop 2: Differential expression](https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/02_differential_expression.ipynb)** — find *which* genes change.